<a href="https://colab.research.google.com/github/pearl-yu/mist5400fall2026/blob/main/week2/fun_demo_guessing_game_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔮 Build a "10 Questions" game with Gemini

**MIST5400 Fall 2026 — Instructor: Pearl (Peiyan) Yu**

***

In this notebook **you'll build a real game** where **Gemini** tries to guess
what you're thinking — just like the old *10 Questions* / *Akinator* game. Then,
as a bonus, you'll **flip it around** so *you* guess what Gemini is thinking.

You don't need to be a Python expert. We'll build it one small piece at a time,
and every piece uses something from **Week 2**:

| Week-2 concept | Where you'll use it |
|---|---|
| **Functions** (`def`) | `ask_gemini()`, `play_twenty_questions()` |
| **Loops** (`for` / `range`) | asking up to 20 questions |
| **Conditionals** (`if/elif/else`) | "is this a question or a final guess?" |
| **Lists** | storing the questions & answers so far |
| **Strings** | reading Gemini's reply and formatting messages |
| **GenAI in Colab** | talking to the Gemini API |

> ⏱️ Takes about 20–30 minutes. Just run each code cell in order and read the
> notes above it. Where you see **🛠️ Your turn**, try editing the code!


## Step 1 — Get your free Gemini key 🔑

Gemini is the same AI that's built into Colab. To use it from *code*, you need a
free API key.



1. Go to **https://aistudio.google.com/apikey** and sign in with your Google
   account. (Use a gmail other than the UGA email. The UGA account doesn't have access to google AI studio)
2. Click **Create API key** and **copy** it.

> 🔒 **Never** type your key straight into a code cell you plan to share — anyone
> who sees the notebook would see your key. Use the secret manager.

Now run the cell below. 👇


In [ ]:
# @title Step 1 code — install the SDK and connect to Gemini
# Get a FREE key at https://aistudio.google.com/apikey (see instructions above).
!pip install -q google-genai

from google import genai

MODEL = "gemini-3.6-flash"   # a fast, free-tier friendly model

# Grab your key: from Colab's 🔑 secret manager if you saved one, else it asks you.
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
if not api_key:
    import getpass
    api_key = getpass.getpass("Paste your Gemini API key (input hidden): ")

client = genai.Client(api_key=api_key)
print("🤖 Connected to Gemini! Model:", MODEL)


## Step 2 — Say hello to Gemini 👋

Let's make sure it works. The line that matters is
`client.models.generate_content(...)`. We give it a **model** and some
**contents** (our message), and it hands back a reply we read with
`response.text`.


In [ ]:
# @title Step 2 code — say hello to Gemini
# This sends one message ("contents") to the model and prints its reply.
response = client.models.generate_content(
    model=MODEL,
    contents="In one short sentence, introduce yourself as a game host for 20 Questions."
)
print(response.text)


## Step 3 — Make a helper **function** 🧩

We'll be talking to Gemini over and over, so let's not retype that whole line
each time. This is *exactly* why **functions** exist (Week 2!). We'll write one
called `ask_gemini(prompt)` that takes a message and returns Gemini's answer.


In [ ]:
# @title Step 3 code — wrap it in a FUNCTION (Week-2 concept!)
# Instead of retyping that whole call every time, we define it ONCE.
def ask_gemini(prompt):
    """Send `prompt` to Gemini and return its text answer."""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text.strip()

# Try it:
print(ask_gemini("Name one animal that lives in the ocean. Just the animal."))


## Step 4 — How will the game actually work? 🎯

Here's the plan, in plain English:

1. Keep a **list** called `history` of every question Gemini asked and how you
   answered.
2. **Loop** up to 20 times. Each time, send Gemini the history and ask for its
   next move.
3. Gemini replies with either **`QUESTION: ...`** or **`GUESS: ...`**. We use an
   **if/else** to tell them apart (that's **string** handling + **conditionals**).
4. If it's a guess and you say *yes* → Gemini wins. If we hit 20 with no correct
   guess → you win!

The trick that makes this work is **telling Gemini the rules clearly** in the
prompt. That's the next step.


## Step 5 — Tell Gemini the rules (a.k.a. *prompt engineering*) 📜

An AI does what you ask — so we have to ask *precisely*. The function below
builds a full instruction that:
- explains the game,
- says "ask ONE yes/no question at a time",
- and forces the reply into a tidy format (`QUESTION:` or `GUESS:`) so our code
  can read it reliably.

Run it and read the printed prompt — that's literally what we send to Gemini.


In [ ]:
# @title Step 5 code — teach Gemini the rules (prompt engineering)
def build_master_prompt(history):
    """Turn the questions-so-far into a full instruction for Gemini."""
    rules = (
        "We are playing 20 Questions. The user is thinking of a person, place, "
        "or thing, and YOU must figure out what it is.\n"
        "Rules:\n"
        "- Ask exactly ONE yes/no question at a time.\n"
        "- Use all previous answers to narrow it down.\n"
        "- When you are fairly confident, make a guess instead of a question.\n"
        "- Reply in EXACTLY one of these two formats (nothing else):\n"
        "    QUESTION: <your single yes/no question>\n"
        "    GUESS: <your single best guess>\n"
    )
    if history:
        convo = "\n".join("Q: %s  ->  A: %s" % (q, a) for q, a in history)
    else:
        convo = "(no questions asked yet)"
    return rules + "\nAnswers so far:\n" + convo + "\n\nYour move:"

# Peek at what the very first prompt looks like:
print(build_master_prompt([]))


## Step 6 — Put it all together 🎮

Now we combine everything: the **loop**, the **conditionals**, the **list**, and
our two functions. Read the comments — you'll see each Week-2 concept labeled.
Run this cell, then run the **PLAY** cell and think of something!

> 💡 When you run the play cell, a little **text box** appears under the cell —
> type your answer there and press **Enter**.


In [ ]:
# @title Step 6 code — the full game loop (run me, then play!)
def play_twenty_questions(max_questions=20):
    print("🧠 Think of a person, place, or thing. I'll try to guess it!")
    print("   (Answer each question with: yes / no / maybe)\n")

    history = []                              # a LIST of (question, answer) pairs
    for q_num in range(1, max_questions + 1):  # a LOOP, up to 20 rounds
        reply = ask_gemini(build_master_prompt(history))

        # Gemini either asks a QUESTION or makes a GUESS — we check which (CONDITIONALS)
        if reply.upper().startswith("GUESS"):
            guess = reply.split(":", 1)[1].strip() if ":" in reply else reply
            ans = input("🤖 Guess #%d — Is it: %s ?  (yes/no) " % (q_num, guess))
            if ans.strip().lower().startswith("y"):
                print("\n🎉 I got it in %d questions! I win!" % q_num)
                return
            history.append(("Is it %s?" % guess, "no"))
        else:
            question = reply.split(":", 1)[1].strip() if ":" in reply else reply
            ans = input("❓ Q%d: %s  (yes/no/maybe) " % (q_num, question))
            history.append((question, ans.strip()))

    # If the loop finishes without a correct guess:
    print("\n🏳️ You stumped me! I'm out of questions.")
    mystery = input("What were you thinking of? ")
    print("Ahh, %s! I'll get it next time. 😄" % mystery)

print("✅ Game ready. Run the next cell to play!")


In [ ]:
# @title ▶️ PLAY: Gemini guesses what YOU'RE thinking
play_twenty_questions(max_questions=10)

## Step 7 — 🔄 Bonus: flip the game (you guess Gemini's secret)

Now the reverse: **Gemini** secretly thinks of something, and **you** ask the
yes/no questions. Two new functions do the work:
- `gemini_picks_secret()` — Gemini chooses an item and we store it (without
  printing it!).
- `answer_about_secret(secret, question)` — Gemini answers your question
  truthfully but won't reveal the item.

When you think you know it, type **`guess: your answer`**.


In [ ]:
# @title Step 7 code — BONUS: flip it! Gemini thinks, YOU guess
def gemini_picks_secret(category="a common animal, food, or household object"):
    """Ask Gemini to secretly choose one item and return just its name."""
    prompt = ("Pick ONE specific secret item that is %s. "
              "Reply with ONLY the item's name, nothing else." % category)
    return ask_gemini(prompt).splitlines()[0].strip()

def answer_about_secret(secret, question):
    """Gemini answers a yes/no question about the secret WITHOUT revealing it."""
    prompt = ("The secret item is: %s.\n"
              "Answer this yes/no question about it truthfully and briefly "
              "(reply yes / no / sometimes). Do NOT reveal the item's name.\n"
              "Question: %s" % (secret, question))
    return ask_gemini(prompt)

def play_mind_reader(max_questions=20):
    secret = gemini_picks_secret()           # stored, but not printed — it's a secret!
    print("🔮 I'm thinking of something. Ask me up to %d yes/no questions." % max_questions)
    print("   When ready, type:  guess: <your answer>\n")

    for q_num in range(1, max_questions + 1):
        q = input("Your question #%d: " % q_num).strip()
        if q.lower().startswith("guess:"):
            g = q.split(":", 1)[1].strip()
            if g.lower() == secret.lower():
                print("\n🎉 YES! It was '%s'. You read my mind in %d moves!" % (secret, q_num))
                return
            print("❌ Not '%s' — keep going!" % g)
            continue
        print("🤖", answer_about_secret(secret, q))

    print("\n⏰ Out of questions! The secret was: '%s'." % secret)

print("✅ Reverse mode ready. Run the next cell to play!")


In [ ]:
# @title ▶️ PLAY: you guess what GEMINI is thinking
play_mind_reader(max_questions=10)


## 🏁 Wrap-up

You just built a working AI game using only Week-2 Python — functions, loops,
conditionals, lists, and strings — plus the Gemini API. 🎉

**Ideas to keep going:**
- Add a **scoreboard** that remembers wins across rounds (a variable + a loop).
- Let the player pick a **category** at the start (person / place / thing).
- Make Gemini's questions funnier by adding a personality line to the rules.

### 📤 Submitting
**File → Download → Download .ipynb** and submit it to
**eLC → In-class exercise**. (Make sure your game cells show some output — do a
**Runtime → Restart and run all** first, but note the `input()` cells need you
to answer them.)

***
*\[The End\] — happy guessing!*
